# 01 — Chunking Strategy Ablation

This notebook compares five chunking strategies across recall@5, recall@10, MRR, nDCG@10, and latency:
- **Fixed** — character-level fixed-size windows
- **Recursive** — LangChain recursive splitter
- **Semantic** — embedding-similarity boundary detection
- **Hierarchical** — LlamaIndex HierarchicalNodeParser
- **Late** — Jina long-context late chunking

In [ ]:
import subprocess, json, pathlib
import pandas as pd
import matplotlib.pyplot as plt

# Run the ablation (edit IDs and n for full run)
ARXIV_IDS = ["2312.10997", "2310.01558"]
OUTPUT = pathlib.Path("../eval_results/chunking_ablation_nb.json")

result = subprocess.run(
    ["python", "-m", "production_rag.evaluation.chunking_ablation",
     "--output", str(OUTPUT)] + ARXIV_IDS,
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-2000:])

In [ ]:
data = json.loads(OUTPUT.read_text())
df = pd.DataFrame(data).set_index("strategy")
df

In [ ]:
metrics = ["recall_at_5", "recall_at_10", "mrr", "ndcg_at_10"]
ax = df[metrics].plot(kind="bar", figsize=(12, 5), rot=0)
ax.set_title("Chunking Strategy Comparison")
ax.set_xlabel("Strategy")
ax.set_ylabel("Score")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("../eval_results/chunking_ablation.png", dpi=150)
plt.show()

In [ ]:
# Latency comparison
ax2 = df[["avg_latency_ms"]].plot(kind="barh", figsize=(8, 4), legend=False)
ax2.set_title("Average retrieval latency (ms) per strategy")
ax2.set_xlabel("Latency (ms)")
plt.tight_layout()
plt.show()